In [3]:
import pandas as pd

#### Load file

In [4]:
df = pd.read_csv("hate_speech_raw_rows_columns.csv")

#### Checking Dataset

In [5]:
df.head()

,record_id,post_id,post_tokens,annotator_labels,annotator_targets,rationales,annotator_count
0,1179055004553900032_twitter,1179055004553900032_twitter,i dont think im getting my baby them white 9 h...,normal | normal | normal,None | None | None,[],3
1,1179063826874032128_twitter,1179063826874032128_twitter,we cannot continue calling ourselves feminists...,normal | normal | normal,None | None | None,[],3
2,1178793830532956161_twitter,1178793830532956161_twitter,nawt yall niggers ignoring me,normal | normal | hatespeech,African | None | African,[],3
3,1179088797964763136_twitter,1179088797964763136_twitter,<user> i am bit confused coz chinese ppl can n...,hatespeech | offensive | hatespeech,Asian | Asian | Asian,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3
4,1179085312976445440_twitter,1179085312976445440_twitter,this bitch in whataburger eating a burger with...,hatespeech | hatespeech | offensive,"Caucasian, Women | Women, Caucasian | Women, C...","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3


#### Checking Datasize

In [6]:
print("Number of rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Number of rows and columns: (20148, 7)

Column names:
['record_id', 'post_id', 'post_tokens', 'annotator_labels', 'annotator_targets', 'rationales', 'annotator_count']


In [7]:
df.isnull().sum()

record_id            0
post_id              0
post_tokens          0
annotator_labels     0
annotator_targets    0
rationales           0
annotator_count      0
dtype: int64

#### Text Preprocessing

#### import libraries

In [8]:
import re
from collections import Counter

#### convert annotator label to one class

In [9]:
def get_majority_label(labels):
    labels = [
        label.strip().lower()
        for label in labels.split("|")
        if label.strip().lower() in ["hatespeech", "offensive", "normal"]
    ]

    if not labels:
        return None

    counts = Counter(labels)
    most_common = counts.most_common()

  
    if len(most_common) > 1 and most_common[0][1] == most_common[1][1]:
        return None

    return most_common[0][0]


df["class"] = df["annotator_labels"].apply(get_majority_label)

print(df[["post_tokens", "annotator_labels", "class"]].head(10))

                                         post_tokens  \
0  i dont think im getting my baby them white 9 h...   
1  we cannot continue calling ourselves feminists...   
2                      nawt yall niggers ignoring me   
3  <user> i am bit confused coz chinese ppl can n...   
4  this bitch in whataburger eating a burger with...   
5  laura loomer raped me while screaming at me in...   
6  and this is why i end up with nigger trainee d...   
7                  nogs jews and dykes how enriching   
8  guilty until proven innocent unless you are a ...   
9  tired i can not support abortion from a moral ...   

                       annotator_labels       class  
0              normal | normal | normal      normal  
1              normal | normal | normal      normal  
2          normal | normal | hatespeech      normal  
3   hatespeech | offensive | hatespeech  hatespeech  
4   hatespeech | hatespeech | offensive  hatespeech  
5  hatespeech | hatespeech | hatespeech  hatespeech  
6   h

#### Clean The Dataset

In [10]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<user>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["cleaned_text"] = df["post_tokens"].apply(clean_text)

df[["post_tokens", "cleaned_text"]].head()

,post_tokens,cleaned_text
0,i dont think im getting my baby them white 9 h...,i dont think im getting my baby them white he ...
1,we cannot continue calling ourselves feminists...,we cannot continue calling ourselves feminists...
2,nawt yall niggers ignoring me,nawt yall niggers ignoring me
3,<user> i am bit confused coz chinese ppl can n...,i am bit confused coz chinese ppl can not acce...
4,this bitch in whataburger eating a burger with...,this bitch in whataburger eating a burger with...


#### Checking Class Distribution

In [11]:
print(df["class"].value_counts(dropna=False))

class
normal        7814
hatespeech    5935
offensive     5480
NaN            919
Name: count, dtype: int64


#### Prepare The Dataset for TF-IDF

In [12]:
df_model = df.dropna(subset=["class"]).copy()

print("Dataset shape:", df_model.shape)
print(df_model["class"].value_counts())

Dataset shape: (19229, 9)
class
normal        7814
hatespeech    5935
offensive     5480
Name: count, dtype: int64


#### import libraries

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

#### seperating input texts and output labels

In [14]:
X = df_model["cleaned_text"]
y = df_model["class"]

print("Input text shape:", X.shape)
print("Output labels shape:", y.shape)

Input text shape: (19229,)
Output labels shape: (19229,)
